# 3 — Product Artifact Diagnostics and Decision Mindmap

Use this notebook after a product run.

Provide either:

- the product artifact directory; or
- the path of any file inside that artifact directory.

The notebook reconstructs the **observable agent process**:

```text
submitted input
→ product interpretation
→ uncertainties
→ search route
→ candidate investigations
→ text and visual evidence
→ acceptance/rejection rules
→ manufacturer-versus-retailer choice
→ final URL
```

It displays recorded evidence, actions, business rules, judgments and conclusions. It does not expose hidden chain-of-thought.


In [ ]:
from __future__ import annotations

import importlib
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "docker-compose.yml").is_file() and (candidate / "src" / "product_evidence_harness").is_dir():
            return candidate
    raise RuntimeError("Could not locate the web_search_tool repository root")

PROJECT_ROOT = find_project_root()
for required in (str(PROJECT_ROOT), str(PROJECT_ROOT / "src")):
    if required not in sys.path:
        sys.path.insert(0, required)
importlib.invalidate_caches()

from product_evidence_harness.artifact_diagnostics import (
    build_artifact_diagnostics,
    plot_artifact_mindmap,
    plot_business_judgement_timeline,
    write_artifact_diagnostic_report,
)

pd.set_option("display.max_columns", 160)
pd.set_option("display.max_colwidth", 240)
print(f"Repository: {PROJECT_ROOT}")
print("This notebook works offline from an existing artifact; the agent service is not required.")


## Artifact path

Paste the directory path or any file path inside the product artifact.


In [ ]:
ARTIFACT_PATH = PROJECT_ROOT / "data" / "artifacts" / "ROW-001"
RUN_DIAGNOSTICS = False

print(ARTIFACT_PATH)


In [ ]:
if RUN_DIAGNOSTICS:
    diagnostics = build_artifact_diagnostics(ARTIFACT_PATH)
    print(f"Loaded artifact: {diagnostics.artifact_dir}")
else:
    print("Ready. Set ARTIFACT_PATH and RUN_DIAGNOSTICS = True.")


# Executive artifact conclusion


In [ ]:
if "diagnostics" not in globals():
    raise RuntimeError("Run the artifact-loading cell first")

display(diagnostics.overview_df)
display(diagnostics.product_input_df)
display(diagnostics.visual_evidence_summary_df)


# Complete decision mindmap

This map groups the complete run into input, identity, search, validation, visual evidence, source authority, observable judgments and final outcome.


In [ ]:
fig = plot_artifact_mindmap(
    diagnostics,
    max_judgement_steps=12,
)
plt.show()


# Chronological decision trace

This is the ordered evidence-to-action sequence that a human reviewer can compare against their own process.


In [ ]:
fig = plot_business_judgement_timeline(
    diagnostics,
    max_steps=20,
)
plt.show()


In [ ]:
judgement_columns = [
    column for column in (
        "sequence_number",
        "decision_stage",
        "business_question",
        "evidence_considered",
        "evidence_sources",
        "visual_evidence_used",
        "visual_evidence_details",
        "agent_judgement",
        "judgement_status",
        "alternatives_considered",
        "alternative_rejected",
        "rejection_reason",
        "business_rule_applied",
        "effect_on_next_action",
        "confidence",
        "final_outcome",
    )
    if column in diagnostics.business_judgement_steps_df
]
display(diagnostics.business_judgement_steps_df[judgement_columns])


# Search, candidate and feature evidence


In [ ]:
display(diagnostics.search_steps_df)

candidate_columns = [
    column for column in (
        "url",
        "source_role",
        "source_tier_name",
        "validation_status",
        "identity_status",
        "identity_accepted",
        "browser_openable",
        "scrapable",
        "coverage",
        "strict_selected",
        "review_selected",
        "decision_reasons",
        "rejection_reasons",
    )
    if column in diagnostics.candidates_df
]
display(diagnostics.candidates_df[candidate_columns or list(diagnostics.candidates_df.columns)])

feature_columns = [
    column for column in (
        "url",
        "source_role",
        "feature_id",
        "feature_name",
        "value",
        "status",
        "confidence",
        "extraction_method",
        "evidence_location",
        "evidence_text",
    )
    if column in diagnostics.feature_evidence_df
]
display(diagnostics.feature_evidence_df[feature_columns or list(diagnostics.feature_evidence_df.columns)])


# Belief changes and evidence ledger


In [ ]:
display(diagnostics.belief_updates_df)
display(diagnostics.evidence_ledger_df)


# Artifact inventory

This table explains every file available for deeper investigation.


In [ ]:
display(diagnostics.artifact_inventory_df)


# Generate a shareable diagnostic report and workbook

The notebook writes `artifact_diagnostic_report.md` and `artifact_diagnostic_workbook.xlsx` into the same product artifact directory. The Markdown report includes a Mermaid decision flow, judgment table, visual-evidence summary, candidate decisions and human review prompts.


In [ ]:
diagnostic_report_path = write_artifact_diagnostic_report(
    diagnostics,
    output_path=diagnostics.artifact_dir / "artifact_diagnostic_report.md",
)
diagnostic_workbook_path = diagnostics.artifact_dir / "artifact_diagnostic_workbook.xlsx"

with pd.ExcelWriter(diagnostic_workbook_path, engine="openpyxl") as writer:
    for sheet_name, frame in diagnostics.tables().items():
        frame.to_excel(writer, sheet_name=sheet_name.replace("_df", "")[:31], index=False)

print(f"Diagnostic Markdown: {diagnostic_report_path}")
print(f"Diagnostic workbook: {diagnostic_workbook_path}")
print(f"Original human judgment artifact: {diagnostics.artifact_dir / 'business_judgement_review.md'}")


## How to use the result

Ask the reviewer:

1. Is the interpreted product identical?
2. Is the sequence of business judgments identical?
3. Were the same candidates rejected for the same reasons?
4. Was image evidence interpreted correctly?
5. Is the manufacturer-versus-retailer choice identical?
6. What is the first divergent step?

The first divergent step becomes the next development requirement.
